# TRACR Town05 Co-Simulation Attack Demo

This notebook uses the existing TRACR dashboard shell, but the simulation loop is now driven by `V2VCoSimClientMaster.step()`. That means CARLA vehicles are controlled by the project V2V path planner/controller stack, while Simu5G carries normal and injected BSM messages.

The demo launches four vehicles on METS-R road `48 -> -3`. Roads `48`, `-39`, `-0`, `-1`, `-2`, and `-3` are registered as co-simulation roads. The second generated vehicle is the ego and obstacle ghost attack target. The fake obstacle is highlighted only in the dashboard bird-eye image as a small orange-red overlay, so it is not visible to the ego vehicle camera.

## Setup

Load the same TRACR support utilities used by `tracr_demo.ipynb`, plus the obstacle ghost attack hook.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import importlib
import os
import sys
import time

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import carla
import tutorials.tracr_demo_support as tracr_demo_support
tracr_demo_support = importlib.reload(tracr_demo_support)

TRACRDashboard = tracr_demo_support.TRACRDashboard
CarlaSensorPanel = tracr_demo_support.CarlaSensorPanel

from clients.V2V_CoSimClient_master import V2VCoSimClientMaster
from cosim_utils.attack_manager import ObstacleGhostVehicleAttack, V2XAttackManager, assign_attack_vehicle_ids
from cosim_utils.helpers import is_port_open, set_random_seed
from utils.carla_util import destroy_carla_actor, open_carla
from utils.simu5g_v2x_util import start_simu5g_bridge_in_terminal
from utils.util import prepare_sim_dirs, read_run_config, run_simulation_in_docker

os.environ["OMNETPP_HOME"] = "/home/siwen/Software/omnetpp-6.1"
os.environ["INET_HOME"] = "/home/siwen/Software/inet-4.5.4"
os.environ["SIMU5G_HOME"] = "/home/siwen/Software/Simu5G"
os.environ["PATH"] = (
    f'{os.environ["OMNETPP_HOME"]}/bin:'
    f'{os.environ["INET_HOME"]}/src:'
    f'{os.environ["SIMU5G_HOME"]}/src:'
    + os.environ.get("PATH", "")
)

print("Repository root:", REPO_ROOT)

Repository root: /home/siwen/Software/METS-R_docker/METS-R_HPC


## Launch services

This cell starts/reuses Simu5G, CARLA, and METS-R, then creates a `V2VCoSimClientMaster`. The four trips are generated sequentially on `48 -> -3`, and the second vehicle is locked as the dashboard ego/attack target.

In [2]:
VEINS_PORT = 9099
COSIM_ROADS = ["48", "-39", "-0", "-1", "-2", "-3"]
VEHICLE_IDS = [1, 2, 3, 4]
EGO_VEHICLE_ID = VEHICLE_IDS[1]
TRIP_SPECS = [(vid, "48", "-3") for vid in VEHICLE_IDS]
TRIP_DEPARTURE_GAP_TICKS = 20
RANDOM_SEED = 42
ATTACK_START_TICK = 20
ATTACK_END_TICK = 280

config = read_run_config("configs/run_cosim_CARLAT5.json")
set_random_seed(RANDOM_SEED, config=config)
config.display_all = False
config.verbose = False
config.enable_debug_draw = False
config.draw_route_plan = False
config.v2v_position_mode = "local"
config.cv2x_communication_range_m = 500.0
config.carla_tick_timeout = 5.0
config.metsr_tick_timeout = 5.0
config.release_queued_cosim_vehicles = True
config.metsr_road = COSIM_ROADS
config.controller_vids = VEHICLE_IDS
config.handoff_spawn_clearance_m = 10.0

if not is_port_open(VEINS_PORT):
    print(f"Starting Simu5G bridge on port {VEINS_PORT}...")
    start_simu5g_bridge_in_terminal(REPO_ROOT, wait_seconds=5.0)
else:
    print(f"Simu5G bridge already listening on port {VEINS_PORT}.")

prepare_sim_dirs(config)

carla_client, carla_tm = open_carla(config)
world = carla_client.get_world()
world.set_weather(carla.WeatherParameters.ClearNoon)
set_random_seed(RANDOM_SEED, traffic_manager=carla_tm)
print("CARLA connected.")

metsr_port = int(config.metsr_port[0] if hasattr(config, "metsr_port") else config.ports[0])
if not is_port_open(metsr_port):
    run_simulation_in_docker(config)
else:
    print(f"METS-R already listening on port {metsr_port}; reusing it.")

cosim_client = V2VCoSimClientMaster(
    config,
    carla_client,
    carla_tm,
    controller_vids=VEHICLE_IDS,
    require_simu5g_uu=True,
)
cosim_client.set_custom_camera(0.0, 0.0, 110.0)

for index, (vid, road_from, road_to) in enumerate(TRIP_SPECS):
    if index > 0 and TRIP_DEPARTURE_GAP_TICKS > 0:
        cosim_client.metsr.tick(TRIP_DEPARTURE_GAP_TICKS, max_wait_seconds=10, poll_timeout=1)
    print(f"Generating trip veh={vid}: {road_from} -> {road_to}")
    cosim_client.metsr.generate_trip_between_roads([vid], road_from, road_to)
    cosim_client.metsr.update_vehicle_sensor_type([vid], "cv2x", True)

attack = ObstacleGhostVehicleAttack(
    target_vehicle_id=EGO_VEHICLE_ID,
    ghost_id=None,
    base_distance_m=6.0,
    ghost_speed_mps=3.0,
    start_tick=cosim_client.current_tick + ATTACK_START_TICK,
    end_tick=cosim_client.current_tick + ATTACK_END_TICK,
    attack_id="tracr_v2v_master_obstacle_ghost",
)
assign_attack_vehicle_ids(attack, VEHICLE_IDS)
attacks = V2XAttackManager([attack])

sensor_panel = CarlaSensorPanel(world, carla, destroy_carla_actor)
sensor_panel.spawn_overhead_camera(z=110.0)
sensor_panel.target_vehicle_id = EGO_VEHICLE_ID
sensor_panel.strict_target = True

runtime = SimpleNamespace(
    config=config,
    metsr=cosim_client.metsr,
    world=world,
    carla_client=carla_client,
    carla_tm=carla_tm,
    carla_state=SimpleNamespace(active_vehicles=cosim_client.carla_vehs, display_vehicles=cosim_client.displayOnly_vehs),
    sensor_panel=sensor_panel,
    generated_vehicle_ids=VEHICLE_IDS,
    v2x_vehicle_ids=VEHICLE_IDS,
    focus_vehicle_id=EGO_VEHICLE_ID,
    lock_focus_vehicle=True,
    bsm_stream_source="simu5g",
    bsm_stream_label="Simu5G + V2V master obstacle ghost attack",
)

{
    "cosim_roads": COSIM_ROADS,
    "trip_specs": TRIP_SPECS,
    "ego_vehicle_id": EGO_VEHICLE_ID,
    "attack": vars(attack),
}

Starting Simu5G bridge on port 9099...
CARLA connected.
Connection established!
Generating trip veh=1: 48 -> -3
Generating trip veh=2: 48 -> -3
Generating trip veh=3: 48 -> -3
Generating trip veh=4: 48 -> -3


{'cosim_roads': ['48', '-39', '-0', '-1', '-2', '-3'],
 'trip_specs': [(1, '48', '-3'),
  (2, '48', '-3'),
  (3, '48', '-3'),
  (4, '48', '-3')],
 'ego_vehicle_id': 2,
 'attack': {'attack_id': 'tracr_v2v_master_obstacle_ghost',
  'start_tick': 80,
  'end_tick': 340,
  'enabled': True,
  'target_vehicle_id': 2,
  'ghost_id': 5,
  'lead_time_s': 0.0,
  'base_distance_m': 6.0,
  'ghost_speed_mps': 3.0}}

## Runtime diagnostics

Use this only when needed. It reports the METS-R queue/co-sim state and the CARLA-managed vehicle IDs from `V2VCoSimClientMaster`.

In [ ]:
print("tick:", cosim_client.current_tick)
print("generated:", VEHICLE_IDS)
print("ego:", EGO_VEHICLE_ID)
print("carla managed:", sorted(cosim_client.carla_vehs.keys()))
print("controllers:", sorted(cosim_client.controllers.keys()))
print("route synced:", dict(cosim_client.route_synced))

print()
print("query_vehicle:")
try:
    print(cosim_client.metsr.query_vehicle(
        id=VEHICLE_IDS,
        private_veh=[True] * len(VEHICLE_IDS),
        transform_coords=True,
    ))
except Exception as exc:
    print("query_vehicle failed:", exc)

print()
print("coSimVehicle:")
print(cosim_client.metsr.query_coSimVehicle())

print()
print("last v2x rows:", len(getattr(cosim_client, "last_v2x_rows", []) or []))
print("last attack injection vehicles:", attacks.last_injection.get("vehicles", []))

## Demo dashboard

Open the printed local dashboard URL. The BSM panel shows the ego vehicle's received Simu5G BSM stream, including the injected obstacle ghost BSM when it is delivered. CARLA also shows an orange-red live marker at the falsified ghost position ahead of ego.

In [ ]:
if not hasattr(runtime, "viz_info"):
    runtime.viz_info = tracr_demo_support._start_viz_with_port_fallback(cosim_client.metsr, {})

dashboard = TRACRDashboard(
    stream_url=runtime.viz_info["url"],
    fullscreen=True,
    bsm_stream_label=runtime.bsm_stream_label,
    bsm_ego_only=True,
    title="V2V Attack Co-Simulation Demo",
)
dashboard_url = dashboard.display_external(port=8899)
dashboard_url

METS-R Vis live stream is available at ws://127.0.0.1:8765; origin=(0.0, 0.0); call render() to send frames.
Serving /home/siwen/Software/METS-R_docker/METS-R_HPC/output/tracr_dashboard with CORS enabled on port 8899...


'http://127.0.0.1:8899/index.html'

127.0.0.1 - - [04/Jul/2026 22:57:40] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:40] "GET /state.json?ts=1783220260395 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:40] "GET /state.json?ts=1783220260897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:41] "GET /state.json?ts=1783220261397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:41] "GET /state.json?ts=1783220261897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:42] "GET /state.json?ts=1783220262397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:42] "GET /state.json?ts=1783220262897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:43] "GET /state.json?ts=1783220263397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:43] "GET /state.json?ts=1783220263897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:44] "GET /state.json?ts=1783220264397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:44] "GET /state.json?ts=1783220264897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:57:45] "GET /state.json?ts=178

## Run the live attack loop

This loop advances the real V2V co-simulation stack with `V2VCoSimClientMaster.step()`, updates the dashboard sensors for the locked ego vehicle, draws the obstacle ghost only on the dashboard bird-eye image as a small orange-red overlay, and refreshes the TRACR dashboard display.

In [ ]:
def _dashboard_state():
    return SimpleNamespace(
        active_vehicles=cosim_client.carla_vehs,
        display_vehicles=cosim_client.displayOnly_vehs,
    )


def _draw_obstacle_ghost_box():
    markers = []
    for vehicle in attacks.last_injection.get("vehicles", []) or []:
        if vehicle.get("role") != "obstacle_ghost_attacker":
            continue
        try:
            location = cosim_client.get_carla_location(vehicle["x"], vehicle["y"])
            location.z += 1.0
            markers.append({"location": location, "size_px": 14})
        except Exception:
            pass
    runtime.sensor_panel.set_overhead_markers(markers)


def run_v2v_master_demo(ticks=1500, sleep_s=0.0, dashboard_every=3, render_every=2):
    last = None
    for step_index in range(int(ticks)):
        predicted_tick = cosim_client.current_tick + 1
        attack_active = any(item.active(predicted_tick) for item in attacks.attacks)
        phase = "obstacle_ghost_attack" if attack_active else "normal"
        step_result = cosim_client.step(extra_v2x_messages=attacks, phase=phase)
        runtime.carla_state = _dashboard_state()
        runtime.sensor_panel.ensure_sensors(runtime.carla_state, preferred_vehicle_ids=[EGO_VEHICLE_ID])
        _draw_obstacle_ghost_box()

        bsm_records = list(step_result.get("v2x", {}).get("stream", []) or [])
        dashboard_step = {
            "state": runtime.carla_state,
            "v2x": step_result.get("v2x", {}),
            "tracr_projection": {"focus_vehicle": EGO_VEHICLE_ID},
        }
        render_info = None
        render_error = None
        if render_every <= 1 or step_index % int(render_every) == 0:
            try:
                render_info = cosim_client.metsr.render(client_wait_timeout=0)
            except Exception as exc:
                render_error = str(exc).splitlines()[0]
        if dashboard is not None and (dashboard_every <= 1 or step_index % int(dashboard_every) == 0):
            dashboard.update(runtime, dashboard_step, bsm_records, render_info=render_info, render_error=render_error)
        last = {
            "tick": step_result.get("tick"),
            "phase": phase,
            "carla_vehicles": sorted(cosim_client.carla_vehs.keys()),
            "controllers": sorted(cosim_client.controllers.keys()),
            "route_synced": dict(cosim_client.route_synced),
            "bsm_count": len(bsm_records),
            "last_attack_vehicles": attacks.last_injection.get("vehicles", []),
        }
        if sleep_s:
            time.sleep(float(sleep_s))
    return last

last_result = run_v2v_master_demo(ticks=1500, sleep_s=0.0, dashboard_every=3, render_every=2)
last_result

[handoff] veh=1 metsr_speed=0.00 spawn_loc=(139.09,-1.98,0.50) spawn_yaw=-179.76 target_velocity=(-10.00,-0.04)
Vehicle 1 entered the co-sim ownership set and is now CARLA-managed.
Vehicle 2 handoff delayed because vehicle 1 is still within 10.0 m of (139.09,-1.98).


127.0.0.1 - - [04/Jul/2026 22:57:52] "GET /state.json?ts=1783220272397 HTTP/1.1" 200 -


[handoff] veh=2 metsr_speed=0.00 spawn_loc=(139.09,-1.98,0.50) spawn_yaw=-179.76 target_velocity=(-10.00,-0.04)
Vehicle 2 entered the co-sim ownership set and is now CARLA-managed.


127.0.0.1 - - [04/Jul/2026 22:57:52] "GET /state.json?ts=1783220272897 HTTP/1.1" 200 -


Vehicle 3 handoff delayed because vehicle 2 is still within 10.0 m of (139.09,-1.98).


...

127.0.0.1 - - [04/Jul/2026 22:58:25] "GET /state.json?ts=1783220305397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:25] "GET /state.json?ts=1783220305897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:26] "GET /state.json?ts=1783220306397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:26] "GET /state.json?ts=1783220306897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:27] "GET /state.json?ts=1783220307397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:27] "GET /state.json?ts=1783220307897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:28] "GET /state.json?ts=1783220308397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:28] "GET /state.json?ts=1783220308897 HTTP/1.1" 200 -


[handoff] veh=4 metsr_speed=0.00 spawn_loc=(139.09,-1.98,0.50) spawn_yaw=-179.76 target_velocity=(-10.00,-0.04)
Vehicle 4 entered the co-sim ownership set and is now CARLA-managed.


127.0.0.1 - - [04/Jul/2026 22:58:29] "GET /state.json?ts=1783220309397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:29] "GET /state.json?ts=1783220309897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:30] "GET /state.json?ts=1783220310397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:30] "GET /state.json?ts=1783220310897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:31] "GET /state.json?ts=1783220311397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:31] "GET /state.json?ts=1783220311897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:32] "GET /state.json?ts=1783220312397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:32] "GET /state.json?ts=1783220312897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:33] "GET /state.json?ts=1783220313397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:33] "GET /state.json?ts=1783220313897 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:34] "GET /state.json?ts=1783220314397 HTTP/1.1" 200 -
127.0.0.1 - - [04/Jul/2026 22:58:34] "GET /

{'tick': 1560,
 'phase': 'normal',
 'carla_vehicles': [],
 'controllers': [],
 'route_synced': {},
 'bsm_count': 0,
 'last_attack_vehicles': []}

Killed


## Manual step mode

Use this cell for a controlled walkthrough. Each call advances one `V2VCoSimClientMaster.step()` and refreshes the dashboard once.

In [ ]:
predicted_tick = cosim_client.current_tick + 1
attack_active = any(item.active(predicted_tick) for item in attacks.attacks)
phase = "obstacle_ghost_attack" if attack_active else "normal"
step_result = cosim_client.step(extra_v2x_messages=attacks, phase=phase)
runtime.carla_state = _dashboard_state()
runtime.sensor_panel.ensure_sensors(runtime.carla_state, preferred_vehicle_ids=[EGO_VEHICLE_ID])
_draw_obstacle_ghost_box()
bsm_records = list(step_result.get("v2x", {}).get("stream", []) or [])
dashboard.update(
    runtime,
    {"state": runtime.carla_state, "v2x": step_result.get("v2x", {}), "tracr_projection": {"focus_vehicle": EGO_VEHICLE_ID}},
    bsm_records,
    render_info=cosim_client.metsr.render(client_wait_timeout=0),
)
{
    "tick": step_result.get("tick"),
    "phase": phase,
    "carla_vehicles": sorted(cosim_client.carla_vehs.keys()),
    "controllers": sorted(cosim_client.controllers.keys()),
    "route_synced": dict(cosim_client.route_synced),
    "bsm_count": len(bsm_records),
    "last_attack_vehicles": attacks.last_injection.get("vehicles", []),
}

## Cleanup

Run this when the demo is done.

In [ ]:
try:
    sensor_panel.close()
except Exception:
    pass
try:
    cosim_client.close()
except Exception as exc:
    print("cleanup warning:", exc)